<a href="https://colab.research.google.com/github/gabrielmprata/anatel_banda_larga_fixa/blob/main/Preprocessamento_banda_larga_fixa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/python/python-original.svg" width="40" height="40"/> <img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/pandas/pandas-original-wordmark.svg" width="40" height="40"/>   <img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

---

>
**Dev**: Gabriel Prata
>
**Data**: 13/05/2026
>
**Última modificação**: 13/05/2026
>
**Contexto**: *Preparar os Dados abertos de banda larga fixa.*
>
---

Pré-processamento dos dados, para serem utilizados em um painel elaborado em Power BI.
>
<img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

>
![Badge em Desenvolvimento](http://img.shields.io/static/v1?label=STATUS&message=EM%20DESENVOLVIMENTO&color=GREEN&style=for-the-badge)

![Badge versao](http://img.shields.io/static/v1?label=Ver.&message=v3.0&color=red&style=for-the-badge&logo=github)

#**<font color=#85d338 size="6"> Import libraries**

In [1]:
# Importação de pacotes
import pandas as pd
import numpy as np
import missingno as ms # para tratamento de missings
import datetime
import calendar
import re # expressão regulares
import unicodedata

#bibliotecas para visualização de dados
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#compactar
from shutil import copyfileobj
import bz2
# Data e hora
from datetime import datetime, date, time

# Configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

#**<font color=#85d338 size="6"> 1. Coleta de dados**

As informações de acessos de **Banda Larga Fixa**, estão no sítio de dados abertos do governo federal, no link abaixo:
>
https://dados.gov.br/dados/conjuntos-dados/acessos---banda-larga-fixa

>
É disponibilizado um arquivo ZIP com todos os anos disponíveis.
>
Para esse projeto o arquivo foi coletado no dia **13/05/2026**.
>
Selecionamos os seguintes arquivos de **2025**, para tratar:
>
```
Acessos_Banda_Larga_Fixa_2025.zip
```
>
E o arquivo de **2024**, para analisar a evolução:
>
```
Acessos_Banda_Larga_Fixa_2024.zip
```

In [41]:
# importando dataset

# Conjunto de dados com os acessos de 2025
df_acessos_bl_2025 = pd.read_csv("Acessos_Banda_Larga_Fixa_2025.csv.bz2", compression='bz2', delimiter=';')

# Conjunto de dados com os acessos de 2024
df_acessos_bl_2024 = pd.read_csv("Acessos_Banda_Larga_Fixa_2024.csv.bz2", compression='bz2', delimiter=';')


No dataset de 2024, só temos interesse no mês de **DEZEMBRO**.
>
Vamos criar um Dataframe selecionado, e depois concatenar com o Dataframe de 2025.

In [42]:
df_acessos_bl_2024 = df_acessos_bl_2024[(df_acessos_bl_2024['Mês'] == 12)].copy()

Utilizei a função "concat" para juntar os DataFranes, em apenas um.

In [43]:
df_acessos_bl = pd.concat([df_acessos_bl_2025, df_acessos_bl_2024],sort=False, ignore_index=True)

In [44]:
# liberar uso de RAM do ambiente
del df_acessos_bl_2024
del df_acessos_bl_2025



---



#**<font color=#85d338 size="6"> 2. Análise de Dados Inicial**

###**<font color=#85d338> 2.1. Estatísticas Descritivas**

Compreende a organização, o resumo e, descrever os dados, que podem ser expressos em tabelas e gráficos.
>
Veremos a seguir alguns comandos para exibir algumas estatísticas descritivas.


In [ ]:
#	Quantidade de atributos e instâncias (linhas/colunas)
df_acessos_bl.shape

(8819006, 16)

<font color=#85d338> Data frame com 16 atributos(features) e cerca de 8.8 milhões de tuplas.
>


---





In [ ]:
# Exibir os 5 primeiros registros
df_acessos_bl.head(5)

,Ano,Mês,Grupo Econômico,Empresa,CNPJ,Porte da Prestadora,UF,Município,Código IBGE Município,Faixa de Velocidade,Velocidade,Tecnologia,Meio de Acesso,Tipo de Pessoa,Tipo de Produto,Acessos
0,2025,12,OUTROS,ARION SERVICES,20059578000120,Pequeno Porte,SP,Guarulhos,3518800,2Mbps a 12Mbps,"12,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,30
1,2025,12,OUTROS,ARION SERVICES,20059578000120,Pequeno Porte,MG,Confins,3117876,> 34Mbps,"100,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,23
2,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Amparo,3501905,12Mbps a 34Mbps,"20,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,3
3,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Borebi,3507456,> 34Mbps,"307,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,1
4,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Iacri,3519204,512kbps a 2Mbps,"2,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,1




---



In [ ]:
# Mostra diversas informações do Dataframe em um único comando, e exibir o uso de memória
df_acessos_bl.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 16 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   Ano                    int64 
 1   Mês                    int64 
 2   Grupo Econômico        object
 3   Empresa                object
 4   CNPJ                   int64 
 5   Porte da Prestadora    object
 6   UF                     object
 7   Município              object
 8   Código IBGE Município  int64 
 9   Faixa de Velocidade    object
 10  Velocidade             object
 11  Tecnologia             object
 12  Meio de Acesso         object
 13  Tipo de Pessoa         object
 14  Tipo de Produto        object
 15  Acessos                int64 
dtypes: int64(5), object(11)
memory usage: 5.8 GB


<font color=#85d338> Data frame com cerca de **6 gigas de memória**.


---

In [ ]:
# Quantidade de valores únicos
df_acessos_bl.nunique()

,0
Ano,2
Mês,12
Grupo Econômico,24
Empresa,18830
CNPJ,11868
Porte da Prestadora,2
UF,27
Município,5297
Código IBGE Município,5570
Faixa de Velocidade,5




---



In [ ]:
# Quantidade de NaN/Missing/Nulls no dataframe
df_acessos_bl.isnull().sum()

,0
Ano,0
Mês,0
Grupo Econômico,205
Empresa,205
CNPJ,0
Porte da Prestadora,205
UF,0
Município,0
Código IBGE Município,0
Faixa de Velocidade,0




---



###**<font color=#85d338> 2.2. Distribuição dos atributos**

>Nessa etapa, iremos verificar a distribuição dos principais atributos. Para ver se existe a necessidade de tomar alguma ação de transformações na etapa de preparação de dados.


---

In [ ]:
df_acessos_bl.describe().round(2)

,Ano,Mês,CNPJ,Código IBGE Município,Acessos
count,8819006.00,8819006.00,8.819006e+06,8819006.00,8819006.00
mean,2024.92,6.95,2.688351e+13,3439790.39,79.74
std,0.26,3.61,2.450851e+13,906882.88,1579.75
min,2024.00,1.00,5.727400e+10,1100015.00,1.00
25%,2025.00,4.00,6.346446e+12,2919306.00,1.00
50%,2025.00,7.00,1.794785e+13,3505708.00,3.00
75%,2025.00,10.00,4.043254e+13,4203907.00,13.00
max,2025.00,12.00,9.755455e+13,5300108.00,666982.00




---



#**<font color=#85d338 size="6"> 3. Pré-Processamento de dados**

Após coletar e analisar os dados na etapa anterior, agora é o momento
de limpar, transformar e apresentar melhor os dados.
>
Assim poderemos obter, na próxima etapa, os melhores resultados possíveis nos algoritmos de
Machine Learning, ou simplesmente apresentar dados mais confiáveis para os clientes em soluções de
business intelligence.


---

###**<font color=#85d338> 3.1. Limpeza**

De forma resumida, a limpeza consiste na verificação da consistência das informações, correção de possíveis erros de preenchimento ou eliminação de valores desconhecidos, redundantes ou não pertencentes ao domínio.



####**<font color=#85d338> 3.1.1 Padronização de dados**

Dentro da programação, possuímos alguns padrões de escrita para nomes de variáveis, funções, classes e assim por diante.
>
Esses padrões de escrita são chamados de estilos de case.
>
Existem diversos tipos de case, nesse projeto iremos utilizar:
>
**Snake Case (snake_case)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).
>
**Remover acentos (remover_acentos)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).

In [45]:
def remover_acentos(texto):
    # Decompõe caracteres acentuados (ex: á -> a + ´)
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # Remove os caracteres de acentuação (intervalo unicode)
    return re.sub(r'[\u0300-\u036f]', '', texto_normalizado)

df_acessos_bl.columns = [remover_acentos(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

Index(['Ano', 'Mes', 'Grupo Economico', 'Empresa', 'CNPJ',
       'Porte da Prestadora', 'UF', 'Municipio', 'Codigo IBGE Municipio',
       'Faixa de Velocidade', 'Velocidade', 'Tecnologia', 'Meio de Acesso',
       'Tipo de Pessoa', 'Tipo de Produto', 'Acessos'],
      dtype='object')

Aplicar **Snake Case**

In [46]:
# Criar uma função para aplicar o snake_case
def snake_case(string):
    string = re.sub(" +", " ", string)   # substitui múltiplos espaços por um espaço
    string = re.sub(" ", "_", string)    # substitui espaço por _
    return string.lower() # transforma em minuscula

df_acessos_bl.columns = [snake_case(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

Index(['ano', 'mes', 'grupo_economico', 'empresa', 'cnpj',
       'porte_da_prestadora', 'uf', 'municipio', 'codigo_ibge_municipio',
       'faixa_de_velocidade', 'velocidade', 'tecnologia', 'meio_de_acesso',
       'tipo_de_pessoa', 'tipo_de_produto', 'acessos'],
      dtype='object')

####**<font color=#85d338> 3.1.2 Redundâncias**

Vamos eliminar as colunas que não iremos utilizar em nossas analises.
>
A conta de estudante no Power BI, suporta apenas arquivos de 1Gb, sendo assim, terei que excluir as informações de município e depois agregar para reduzir a quantidade de linhas.
>
A ideia é ter um dataframe mais leve, e com pouco espaço em disco.

In [47]:
df_acessos_bl.drop([
				            'cnpj','tecnologia'
                    ,'tipo_de_pessoa'
                    ,'tipo_de_produto'
                    ,'municipio'
                    ,'codigo_ibge_municipio'
                    ], axis=1, inplace= True)

In [48]:
df_acessos_bl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 10 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   ano                  int64 
 1   mes                  int64 
 2   grupo_economico      object
 3   empresa              object
 4   porte_da_prestadora  object
 5   uf                   object
 6   faixa_de_velocidade  object
 7   velocidade           object
 8   meio_de_acesso       object
 9   acessos              int64 
dtypes: int64(3), object(7)
memory usage: 672.8+ MB


####**<font color=#85d338> 3.1.3 Tratamento de Missings**

Como o DataFrame tem muitos atributos, e muitos deles possuem valores nulos, vou utlizar um método para mostrar apenas os valores nulos e o percentual.
>
1) Converter a função isnull() em um pd.series e depois transformar em DF, assim conseguimos filtrar quais atributos são nulos.
>

In [ ]:
#1)
# cria um pd.series
dfnull = df_acessos_bl.isnull().sum()

# Converte series em dataframe
dfnull = (dfnull.to_frame(name="QTD"))

tot = len(df_acessos_bl) #Total de registros no dataset

#Criano o atributo perc, para saber o percentual de registros nulo do atributo
dfnull["perc"] = ((dfnull['QTD']/tot)*100).round(2)

#Mostrar apenas os atributos com valores nulos, ordenando para o com mais linhas nulas
dfnull.query('QTD > 0').sort_values(by='perc', ascending=False)

,QTD,perc
grupo_economico,205,0.0
empresa,205,0.0
porte_da_prestadora,205,0.0


In [49]:
df_acessos_bl['grupo_economico'] = df_acessos_bl['grupo_economico'].fillna('OUTROS')
df_acessos_bl['empresa'] = df_acessos_bl['empresa'].fillna('Não identificado')
df_acessos_bl['porte_da_prestadora'] = df_acessos_bl['porte_da_prestadora'].fillna('Pequeno Porte')




---



###**<font color=#85d338> 3.2 Criação de recursos**

Também conhecida como ***feature engineering***, a criação de recursos consiste em criar, a partir dos atributos originais, um conjunto de atributos que capture informações importantes.

####**<font color=#85d338> 3.2.1 Data completa**

Criar um atributo do tipo DATE TIME para o Power BI criar a hierarquia de datas.

In [50]:
df_acessos_bl['ano_mes'] = df_acessos_bl['ano'].map(str) + '-' + df_acessos_bl['mes'].map(str)
df_acessos_bl['ano_mes'] = pd.to_datetime(df_acessos_bl['ano_mes'])
df_acessos_bl['ano_mes'] = df_acessos_bl['ano_mes'].dt.strftime('%Y-%m')

In [52]:
df_acessos_bl['data'] = pd.to_datetime(df_acessos_bl['ano_mes'])

In [53]:
# excluir atributo
df_acessos_bl.drop(['ano_mes'], axis=1, inplace= True)



---



###**<font color=#85d338> 3.3 Redução da dimensionalidade**

####**<font color=#85d338> 3.3.1 Agregação**

Também pode ser considerada uma técnica de redução de dimensionalidade, pois reduz o número de colunas e linhas do dataset.
>
O nosso dataset, é aberto por Municipio, e nesse momento não iremos analisar nessa granularidade, pois a conta gratuita do Power BI, permite arquivos de até 1Gb.
>
Vamos criar um dataset agregado com a visão por UF, agrupandos os atributos que não se repetem.
>
Assim, será possivel fazer analises mais rápidas, e ter um dataset menor para usar no Github e no Power BI.

In [54]:
df_acessos_bl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 11 columns):
 #   Column               Dtype         
---  ------               -----         
 0   ano                  int64         
 1   mes                  int64         
 2   grupo_economico      object        
 3   empresa              object        
 4   porte_da_prestadora  object        
 5   uf                   object        
 6   faixa_de_velocidade  object        
 7   velocidade           object        
 8   meio_de_acesso       object        
 9   acessos              int64         
 10  data                 datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 740.1+ MB


In [55]:
# Original, sem criar as faixas
df_acessos_bl_fixa = df_acessos_bl.groupby(["ano",
                                              "mes",
                                            "data",
                                              "grupo_economico",
                                              "empresa",
                                              "porte_da_prestadora",
                                              "uf",
                                              "meio_de_acesso","velocidade",
                                            "faixa_de_velocidade"])['acessos'].sum().reset_index()

In [59]:
df_acessos_bl_fixa.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1683844 entries, 0 to 1683843
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   ano                  1683844 non-null  int64         
 1   mes                  1683844 non-null  int64         
 2   data                 1683844 non-null  datetime64[ns]
 3   grupo_economico      1683844 non-null  object        
 4   empresa              1683844 non-null  object        
 5   porte_da_prestadora  1683844 non-null  object        
 6   uf                   1683844 non-null  object        
 7   meio_de_acesso       1683844 non-null  object        
 8   velocidade           1683844 non-null  object        
 9   faixa_de_velocidade  1683844 non-null  object        
 10  acessos              1683844 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 141.3+ MB


Uma redução de 8.8 milhões de registros para 1.6M .
>
Uso de memória de 740Mb para 141Mb.

Na próxima etapa, iremos exportar o dataset e realizar a criação de novos recursos no Power BI.
>
Assim, poderei aplicar meus conhecimentos em DAX.

###**<font color=#85d338> 3.4 Export**

Agora iremos exportar o dataset em CSV, para usarmos no Power BI

In [57]:
# Exportar para csv
df_acessos_bl_fixa.to_csv('df_acessos_bl_fixa.csv', sep=';', index=False)

# compactar arquivo com nivel de compressão máxima
with open('df_acessos_bl_fixa.csv', 'rb') as input:
    with bz2.BZ2File('df_acessos_bl_fixa.csv.bz2', 'wb', compresslevel=9) as output:
        copyfileobj(input, output)